# Step 1 — Zero-Shot Baseline for Medical VQA

**Goal of this notebook:** find out whether there is real room for fine-tuning to improve, and prove the evaluation harness is trustworthy — *before* spending a single GPU-hour on training.

**Model:** `Qwen/Qwen2.5-VL-3B-Instruct`
**Datasets:** VQA-RAD (radiology), SLAKE (multi-modality)

**The success criterion.** Published zero-shot numbers for this model on SLAKE are roughly **67% closed-ended / 51% open-ended**. Supervised specialists reach ~85–90% closed. So:

| Outcome | Meaning | Action |
|---|---|---|
| We land near 67/51 on SLAKE | Harness is correct, gap is real | **Proceed to training** |
| We land far below | Scoring bug (usually normalisation) | Fix harness first |
| We land near 90% | No headroom | Change dataset or model |

**Runtime:** set to GPU (`Runtime > Change runtime type > T4 GPU`). Expect ~30–60 minutes total.

## 1. Check the GPU

The GPU determines dtype and attention backend. A T4 is Turing and supports **neither bf16 nor FlashAttention-2** — the code handles this automatically, but you should know what you got.

In [ ]:
!nvidia-smi

import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"\nGPU        : {torch.cuda.get_device_name(0)}")
    print(f"Capability : {major}.{minor}")
    print(f"bf16 OK    : {major >= 8}   (False on T4 -> code will use fp16)")
    print(f"FlashAttn2 : {major >= 8}   (False on T4 -> code will use SDPA)")
else:
    print("NO GPU. Runtime > Change runtime type > T4 GPU")

## 2. Mount Drive

Colab sessions die without warning. Everything — model cache and predictions — goes to Drive so a disconnect costs you minutes, not hours.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/medvlm'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Cache HF models on Drive: avoids re-downloading ~6 GB every session.
os.environ['HF_HOME'] = f'{PROJECT_DIR}/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print("Project dir:", PROJECT_DIR)

## 3. Get the code

Two options. **Option A is what you should use** once you have pushed this repo to your own GitHub — it keeps the notebook thin and the code version-controlled.

Option B (upload the folder manually) is fine for the very first run.

In [ ]:
# --- OPTION A: clone your repo (use this once pushed to GitHub) ---
# %cd /content
# !git clone https://github.com/YOUR_USERNAME/medvlm.git
# %cd /content/medvlm

# --- OPTION B: upload the medvlm folder to Drive, then copy it in ---
!cp -r /content/drive/MyDrive/medvlm/code /content/medvlm 2>/dev/null || echo "adjust this path to where you put the code"
%cd /content/medvlm
!ls -la

## 4. Install dependencies

`qwen-vl-utils` is not optional — it is what applies the image-token budget. Restart the runtime if prompted, then re-run from cell 2.

In [ ]:
!pip install -q -r requirements.txt
print("\nInstalled. If Colab asks you to restart, do it, then re-run from section 2.")

## 5. Run the unit tests FIRST

These need no GPU and no downloads. They test the one thing that silently ruins medical VQA evaluations: a base model answering correctly in hedged clinical prose (*"there does not appear to be an acute fracture"*) being scored as wrong against gold `"no"`.

**If these fail, stop.** Every number downstream would be meaningless.

In [ ]:
!python tests/test_normalize_metrics.py

## 6. Inspect the dataset before trusting it

Check the schema, the closed/open balance, and — importantly — the **majority-class baseline**. If 60% of closed answers are "yes", then a model that always says "yes" scores 60%. Your model must beat *that*, not zero.

Dataset ids on the Hub occasionally move. If this cell errors, the message tells you what to do.

In [ ]:
!python -m src.data --inspect vqa_rad --n 5

In [ ]:
# SLAKE has explicit OPEN/CLOSED labels, which is why it is our validation set.
!python -m src.data --inspect slake --n 5

## 7. Run the three baselines

Three runs, and the differences between them are the actual result:

| Run | Prompt | Image | What it measures |
|---|---|---|---|
| **naive** | bare question | yes | raw out-of-the-box behaviour |
| **constrained** | "answer in one word" | yes | **the honest baseline** — best without weight updates |
| **blind** | "answer in one word" | **no** | language-prior floor |

Your fine-tuned model must beat the **constrained** number. Beating the naive one just proves you can write a prompt.

Each run appends to a `.jsonl` as it goes. If the session dies, re-run the same cell — it resumes.

In [ ]:
# --- Smoke test: 20 examples, ~2 minutes. Confirms the whole path works. ---
!python -m src.evaluate --config configs/baseline_vqarad_constrained.yaml \
    --limit 20 --output_dir /content/drive/MyDrive/medvlm/outputs/smoke

print("\nIf that printed metrics without crashing, the pipeline works end to end.")

In [ ]:
# --- Run 1/3: constrained prompt, full VQA-RAD test set (~450 items) ---
!python -m src.evaluate --config configs/baseline_vqarad_constrained.yaml \
    --output_dir /content/drive/MyDrive/medvlm/outputs/baseline

In [ ]:
# --- Run 2/3: naive prompt ---
!python -m src.evaluate --config configs/baseline_vqarad_naive.yaml \
    --output_dir /content/drive/MyDrive/medvlm/outputs/baseline

In [ ]:
# --- Run 3/3: blind (no image). Fast, since there are no images to encode. ---
!python -m src.evaluate --config configs/baseline_vqarad_blind.yaml \
    --output_dir /content/drive/MyDrive/medvlm/outputs/baseline

## 8. The validation run: SLAKE

**This is the run that decides whether you proceed.** SLAKE is the benchmark with published Qwen2.5-VL zero-shot numbers (~67% closed / ~51% open). Landing near them means the harness is correct.

Larger than VQA-RAD, so allow ~20–40 minutes on a T4.

In [ ]:
!python -m src.evaluate --config configs/baseline_slake_constrained.yaml \
    --output_dir /content/drive/MyDrive/medvlm/outputs/baseline

In [ ]:
# Blind SLAKE too -- SLAKE's explicit closed/open labels make its
# language-prior floor the more meaningful of the two.
!python -m src.evaluate --config configs/baseline_slake_constrained.yaml \
    --blind --batch_size 8 \
    --output_dir /content/drive/MyDrive/medvlm/outputs/baseline

## 9. The decision

This applies the five checks: harness validity, format gap, vision contribution, headroom, and knowledge-vs-format diagnosis.

In [ ]:
!python scripts/compare_baselines.py \
    --output_dir /content/drive/MyDrive/medvlm/outputs/baseline

## 10. Read the actual failures

Aggregate numbers hide the interesting part. Look at *how* the model is wrong — this is where your report's qualitative section comes from, and it often reveals a scoring bug the metrics can't show you.

In [ ]:
import json, pathlib
from src.metrics import exact_match

path = pathlib.Path('/content/drive/MyDrive/medvlm/outputs/baseline')
f = sorted(path.glob('vqa_rad*constrained_seed0.jsonl'))[0]
records = [json.loads(l) for l in f.open() if l.strip()]

wrong = [r for r in records if exact_match(r['prediction'], r['answer']) < 1.0]
print(f"{len(wrong)}/{len(records)} incorrect\n")

for r in wrong[:15]:
    print(f"[{r['answer_type']}] Q: {r['question']}")
    print(f"    gold: {r['answer']!r}")
    print(f"    pred: {r['prediction']!r}\n")

print("Ask yourself: are these genuinely wrong, or is the scorer mis-crediting them?")
print("If more than ~2 in 15 are scoring bugs, fix the normaliser and re-score")
print("(no GPU needed -- the raw predictions are already saved).")

## 11. Save your baseline

Copy the summary table into your README now, while the context is fresh.

---

### Interpreting what you got

**Landed near 67/51 on SLAKE** → harness validated, headroom confirmed. Proceed to Step 2.

**Scored much lower** → almost always normalisation. Read the failure dump in section 10. Re-scoring is free: the predictions are on disk.

**Scored near 90%** → no headroom. Switch to PathVQA (harder) or a smaller base model.

**Blind is close to sighted** → the model barely uses the image. Report this prominently; it is a real finding about the benchmark, and it means your fine-tuning story should focus on grounding.

### What you now have for the repo
- Validated evaluation harness with unit tests
- Three baselines that separate knowledge gaps from format gaps
- A language-prior floor almost no one else reports
- Raw predictions saved for re-scoring without a GPU